# CNN Baseline: OCT-C8 Classification

We'll train a CNN to classify the images from C8 and save it for Grad-CAM analysis

In [ ]:
from pathlib import Path
import sys
import torch


PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# DATA_DIR = PROJECT_ROOT / "data/oct_c8/RetinalOCT_Dataset/RetinalOCT_Dataset"
DATA_DIR = PROJECT_ROOT / "data/RetinalOCT_Dataset"

RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = RESULTS_DIR / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Data exists:", DATA_DIR.exists())
print("\n")
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

In [ ]:
# Check classes
from src.dataset import get_oct_c8_dataloaders

train_loader, val_loader, test_loader, class_names = get_oct_c8_dataloaders(
    DATA_DIR,
    batch_size=32,
    num_workers=0
)

class_names

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)
print(labels[:10])

In [ ]:
from src.models import build_resnet50

num_classes = len(class_names)

model = build_resnet50(
    num_classes=num_classes,
    pretrained=True,
    freeze_backbone=True
)

model = model.to(device)

model.fc

In [ ]:
# check forwards pass
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)

print(outputs.shape)

In [ ]:
import torch.nn as nn
import torch.optim as optim

from src.train import train_model

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=1e-3
)

checkpoint_path = MODEL_DIR / "resnet50_oct_c8_frozen.pt"

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=3,
    checkpoint_path=checkpoint_path
)
# freeze backbone = true, so  this trains only the final classification layer first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))

plt.plot(history_df["epoch"], history_df["train_acc"], marker="o", label="Train")
plt.plot(history_df["epoch"], history_df["val_acc"], marker="o", label="Validation")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Frozen ResNet50 Training Accuracy")
plt.legend()
plt.grid(True)
plt.show()

The first ResNet50 experiment used ImageNet-pretrained weights with the convolutional backbone frozen. Only the final classification layer was trained. After three epochs, validation accuracy reached approximately 85 percent. This indicates that pretrained CNN features transfer reasonably well to retinal OCT classification, even before fine-tuning the deeper convolutional layers.


In [ ]:
from src.train import evaluate

# Reload best checkpoint
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

test_loss, test_acc = evaluate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
import torch
import numpy as np

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)

        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names
    )
)

## Fine-Tuning the Last ResNet Block

The frozen-backbone model achieved good classification accuracy, but Grad-CAM maps were sometimes diffuse. To make the model more OCT-specific, we next fine-tune the final ResNet block (`layer4`) while keeping earlier layers frozen.

In [ ]:
# Load a fresh pretrained model
from src.models import build_resnet50

model_ft = build_resnet50(
    num_classes=len(class_names),
    pretrained=True,
    freeze_backbone=False
)

model_ft = model_ft.to(device)

In [ ]:
# Freeze everything, then unfreeze layer4 and fc
for param in model_ft.parameters():
    param.requires_grad = False

for param in model_ft.layer4.parameters():
    param.requires_grad = True

for param in model_ft.fc.parameters():
    param.requires_grad = True

In [ ]:
# check trainable params
trainable_params = sum(
    p.numel() for p in model_ft.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model_ft.parameters()
)

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Percent trainable: {100 * trainable_params / total_params:.2f}%")

In [ ]:
# define optimizer with a small learning rate
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, model_ft.parameters()),
    lr=1e-5
)

In [ ]:
from src.train import train_model

checkpoint_path_ft = MODEL_DIR / "resnet50_oct_c8_layer4_finetuned.pt"

history_ft = train_model(
    model=model_ft,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_ft,
    device=device,
    num_epochs=3,
    checkpoint_path=checkpoint_path_ft
)

In [ ]:
# plot fine tuning history
history_ft_df = pd.DataFrame(history_ft)

plt.figure(figsize=(8, 5))

plt.plot(
    history_ft_df["epoch"],
    history_ft_df["train_acc"],
    marker="o",
    label="Train"
)

plt.plot(
    history_ft_df["epoch"],
    history_ft_df["val_acc"],
    marker="o",
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fine-Tuned ResNet50 Training Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# evaluation on test set
model_ft.load_state_dict(
    torch.load(checkpoint_path_ft, map_location=device)
)

test_loss_ft, test_acc_ft = evaluate(
    model=model_ft,
    dataloader=test_loader,
    criterion=criterion,
    device=device
)

print(f"Fine-tuned Test Loss: {test_loss_ft:.4f}")
print(f"Fine-tuned Test Accuracy: {test_acc_ft:.4f}")

In [ ]:
all_preds_ft = []
all_labels_ft = []

model_ft.eval()

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model_ft(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        all_preds_ft.extend(preds)
        all_labels_ft.extend(labels.numpy())

all_preds_ft = np.array(all_preds_ft)
all_labels_ft = np.array(all_labels_ft)

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        all_labels_ft,
        all_preds_ft,
        target_names=class_names
    )
)